In [1]:
import preprocessing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor
from xgboost import XGBRegressor
# https://scikit-learn.org/stable/auto_examples/applications/plot_time_series_lagged_features.html

In [2]:
df_op = preprocessing.preprocessing_operating_data()

/Users/kamalrajanisrani/Documents/MSc Advanced Machine Learning/Semester 1/Applied Machine Learning/Coursework/Applied/preprocessing.py:7: DtypeWarning: Columns (2,3,4,5,6,7,9,10,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df_op = pd.read_csv(r'4000 series operating data.csv')


In [3]:
df_p = preprocessing.preprocessing_product_data()

In [4]:
df_p.groupby('Batch')['Product (g/litre)'].mean()

Batch
4030    24.511466
4032    24.623969
4033    24.800353
4034    25.052205
4035    24.647115
4036    24.449189
4037    23.710458
4038      23.8392
4039    23.869182
4040    26.450455
4041     27.26327
4042    26.367835
4043    27.041068
4044    25.463488
4045    26.591552
4046    26.017734
4047    26.370833
4048     23.77087
4050    21.477518
4051    22.792512
4052    25.787324
Name: Product (g/litre), dtype: object

In [5]:
df_product_rate_per_batch = (
    df_op.groupby('Batch')['TOTAL LIQUID INFLOW'].mean() * df_p.groupby('Batch')['Product (g/litre)'].mean()
    *
    0.001)

In [6]:
df_product_rate_per_batch

Batch
4030    616.305179
4032    621.162886
4033    608.869223
4034    622.196071
4035    606.416985
4036    609.172327
4037    568.419365
4038    593.373544
4039    581.763278
4040     651.30302
4041    670.229408
4042    645.844169
4043    669.367477
4044    621.672935
4045    640.864621
4046    644.787017
4047    663.082829
4048    596.268039
4050    536.552036
4051    549.661318
4052    629.353928
4053           NaN
dtype: object

In [7]:
df_product_rate_per_batch.sort_values(ascending=False)

Batch
4041    670.229408
4043    669.367477
4047    663.082829
4040     651.30302
4042    645.844169
4046    644.787017
4045    640.864621
4052    629.353928
4034    622.196071
4044    621.672935
4032    621.162886
4030    616.305179
4036    609.172327
4033    608.869223
4035    606.416985
4048    596.268039
4038    593.373544
4039    581.763278
4037    568.419365
4051    549.661318
4050    536.552036
4053           NaN
dtype: object

In [8]:
df_op.columns

Index(['Date and time', 'Batch', 'LIQUID INFLOW 1', 'LIQUID INFLOW 2',
       'LIQUID INFLOW 3', 'LIQUID INFLOW 4', 'LIQUID INFLOW 5',
       'LIQUID INFLOW 6', 'pH', 'GAS INFLOW 1', 'GAS INFLOW 2', 'GAS INFLOW 3',
       'GAS INFLOW 4', 'OFFGAS 1', 'OFFGAS 2', 'PRESSURE 1', 'PRESSURE 2',
       'OXYGEN', 'TOTAL LIQUID INFLOW'],
      dtype='object')

In [9]:
df_op = preprocessing.add_product_to_operating(df_op, df_p)

In [10]:
df_op

,Date and time,Batch,LIQUID INFLOW 1,LIQUID INFLOW 2,LIQUID INFLOW 3,LIQUID INFLOW 4,LIQUID INFLOW 5,LIQUID INFLOW 6,pH,GAS INFLOW 1,GAS INFLOW 2,GAS INFLOW 3,GAS INFLOW 4,OFFGAS 1,OFFGAS 2,PRESSURE 1,PRESSURE 2,OXYGEN,TOTAL LIQUID INFLOW,Product (g/litre)
0,2019-02-04 08:46:00,4030,1049.57,25.91,14.44,NaN,297.73,14980.0,5.76,NaN,NaN,NaN,55.14,1.87,16.94,1.79,5.15,6.78,16367.65,NaN
1,2019-02-04 09:01:00,4030,1049.09,25.62,13.54,NaN,357.44,15000.0,5.79,NaN,NaN,NaN,56.88,1.89,20.52,1.80,5.16,8.39,16445.69,NaN
2,2019-02-04 09:16:00,4030,1049.61,25.44,13.59,NaN,356.83,15010.0,5.80,NaN,NaN,NaN,56.03,1.94,23.77,1.80,5.15,8.07,16455.47,NaN
3,2019-02-04 09:31:00,4030,1047.57,25.59,13.97,NaN,356.77,15000.0,5.79,NaN,NaN,NaN,53.91,2.00,27.01,1.80,5.15,7.23,16443.90,19.2
4,2019-02-04 09:46:00,4030,1048.16,25.49,13.43,NaN,357.21,15010.0,5.78,NaN,NaN,NaN,53.97,2.05,30.15,1.80,5.13,7.16,16454.29,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83199,2021-05-25 21:15:00,4053,889.62,0.02,0.01,0.0,0.14,12910.0,5.83,11668.37,33.20,39.50,0.00,0.72,4.46,2.00,5.11,113.78,13799.79,NaN
83200,2021-05-25 21:20:00,4053,888.61,0.02,0.01,0.0,0.07,20210.0,5.72,11283.46,29.42,34.63,0.00,0.68,4.43,1.99,5.09,114.99,21098.71,NaN
83201,2021-05-25 21:25:00,4053,890.29,0.02,0.01,0.0,0.09,20950.0,5.62,10838.39,26.67,30.97,0.00,0.65,4.49,1.99,5.07,115.00,21840.41,NaN
83202,2021-05-25 21:30:00,4053,878.23,0.02,0.01,0.0,0.14,20960.0,5.53,10510.03,24.30,27.70,0.00,0.63,4.48,2.00,5.04,115.00,21838.40,NaN


In [11]:
df_op.to_csv('4000 series operating data product.csv')

In [ ]:
# df_op = pd.read_csv('4000 series operating data product.csv')
# df_op['Date and time'] = pd.to_datetime(df_op['Date and time'])

In [12]:
df_op.columns

Index(['Date and time', 'Batch', 'LIQUID INFLOW 1', 'LIQUID INFLOW 2',
       'LIQUID INFLOW 3', 'LIQUID INFLOW 4', 'LIQUID INFLOW 5',
       'LIQUID INFLOW 6', 'pH', 'GAS INFLOW 1', 'GAS INFLOW 2', 'GAS INFLOW 3',
       'GAS INFLOW 4', 'OFFGAS 1', 'OFFGAS 2', 'PRESSURE 1', 'PRESSURE 2',
       'OXYGEN', 'TOTAL LIQUID INFLOW', 'Product (g/litre)'],
      dtype='object')

In [ ]:
#df_op.drop(columns=['Unnamed: 0'], inplace=True)

In [13]:
df_op = df_op.sort_values(['Batch', 'Date and time'])

In [14]:
for i in range(4):
    df_op[f'LIQUID INFLOW 1_lag{i+1}'] = df_op.groupby('Batch')['LIQUID INFLOW 1'].shift(i+1)
    df_op[f'LIQUID INFLOW 2_lag{i+1}'] = df_op.groupby('Batch')['LIQUID INFLOW 2'].shift(i+1)
    df_op[f'LIQUID INFLOW 3_lag{i+1}'] = df_op.groupby('Batch')['LIQUID INFLOW 3'].shift(i+1)
    df_op[f'LIQUID INFLOW 4_lag{i+1}'] = df_op.groupby('Batch')['LIQUID INFLOW 4'].shift(i+1)
    df_op[f'LIQUID INFLOW 5_lag{i+1}'] = df_op.groupby('Batch')['LIQUID INFLOW 5'].shift(i+1)
    df_op[f'LIQUID INFLOW 6_lag{i+1}'] = df_op.groupby('Batch')['LIQUID INFLOW 6'].shift(i+1)

    df_op[f'GAS INFLOW 1_lag{i+1}'] = df_op.groupby('Batch')['GAS INFLOW 1'].shift(i+1)
    df_op[f'GAS INFLOW 2_lag{i+1}'] = df_op.groupby('Batch')['GAS INFLOW 2'].shift(i+1)
    df_op[f'GAS INFLOW 3_lag{i+1}'] = df_op.groupby('Batch')['GAS INFLOW 3'].shift(i+1)
    df_op[f'GAS INFLOW 4_lag{i+1}'] = df_op.groupby('Batch')['GAS INFLOW 4'].shift(i+1)
    
for i in range(2):
    df_op[f'pH_lag{i+1}'] = df_op.groupby('Batch')['pH'].shift(i+1)
    df_op[f'OFFGAS 1_lag{i+1}'] = df_op.groupby('Batch')['OFFGAS 1'].shift(i+1)
    df_op[f'OFFGAS 2_lag{i+1}'] = df_op.groupby('Batch')['OFFGAS 2'].shift(i+1)
    df_op[f'PRESSURE 1_lag{i+1}'] = df_op.groupby('Batch')['PRESSURE 1'].shift(i+1)
    df_op[f'PRESSURE 2_lag{i+1}'] = df_op.groupby('Batch')['PRESSURE 2'].shift(i+1)
    df_op[f'OXYGEN_lag{i+1}'] = df_op.groupby('Batch')['OXYGEN'].shift(i+1)


In [15]:
df_op['Batch'].unique()

array([4030, 4032, 4033, 4034, 4035, 4036, 4037, 4038, 4039, 4040, 4041,
       4042, 4043, 4044, 4045, 4046, 4047, 4048, 4050, 4051, 4052, 4053])

In [16]:
batch_start = df_op.groupby('Batch')['Date and time'].transform('min')
df_op['time_since_batch_start_hr'] = ((df_op['Date and time'] - batch_start).dt.total_seconds() / 3600)

In [17]:
df_op.groupby('Batch')['Product (g/litre)'].count()

Batch
4030    186
4032    190
4033    169
4034    195
4035    156
4036    185
4037    111
4038    171
4039    110
4040    176
4041    156
4042    187
4043     87
4044     86
4045     58
4046    168
4047     67
4048    207
4050    247
4051    546
4052    552
4053      0
Name: Product (g/litre), dtype: int64

In [18]:
batch_dummies = pd.get_dummies(df_op['Batch'], prefix='Batch', dtype=int)
df_op = pd.concat([df_op, batch_dummies], axis=1)

In [19]:
df_op['Product (g/litre)'] = df_op.pop('Product (g/litre)')

In [20]:
df_op_train = df_op[(df_op['Batch'].between(4030, 4050)) & (df_op['Product (g/litre)'].notna())]

df_op_val = df_op[(df_op['Batch'].isin([4051, 4052])) & (df_op['Product (g/litre)'].notna())]

df_op_test = df_op[df_op['Batch'] == 4053]

In [21]:
print(df_op_train.shape)
print(df_op_val.shape)

(2912, 95)
(1098, 95)


In [22]:
target = 'Product (g/litre)'

drop_cols = ['Date and time', 'Batch', target]
feature_cols = [c for c in df_op_train.columns if c not in drop_cols]

X_train = df_op_train[feature_cols]
y_train = df_op_train[target]

X_val = df_op_val[feature_cols]
y_val = df_op_val[target]

In [23]:
model = XGBRegressor(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=6,
    min_child_weight=7,
    gamma=1,
    subsample=0.7,
    colsample_bytree=0.6,
    reg_alpha=5,
    reg_lambda=11,
    random_state=42
)

In [24]:
model.fit(X_train, y_train)

y_val_pred = model.predict(X_val)
y_train_pred = model.predict(X_train)

print("Validation MAE:", mean_absolute_error(y_val, y_val_pred))
print("Validation R2:", r2_score(y_val, y_val_pred))

Validation MAE: 1.4145299703740464
Validation R2: 0.6213638470846128
